# nb02 — Poison EDA

Characterise the backdoor trigger: what do the 20 poisoned-streak bboxes look like visually
and in terms of model confidence, compared to genuine streak detections on the test set?

**Outputs (saved to `/kaggle/working/`):**
- `poison_crops.png` — contrast-stretched 16-bit crops of the 20 annotated poison bboxes
- `eda_summary.json` — trigger strengths, geometric + intensity stats for both poison and normal
- `docs/poison-eda.md` printed below (copy-paste into the repo)

In [ ]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'

In [ ]:
import json
import math
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor
from tqdm import tqdm

In [ ]:
def _find_comp_data():
    competitions = Path("/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models")
    standard     = Path("/kaggle/input/neural-debris-removal-in-streak-detection-models")
    return competitions if competitions.exists() else standard

COMP_ROOT        = _find_comp_data()
POISONED_WEIGHTS = str(COMP_ROOT / "poisoned_model/poisoned_model.pth")
UNLEARN_DIR      = str(COMP_ROOT / "unlearn_set")
TEST_DIR         = str(COMP_ROOT / "test_set/test_set")
OUT_DIR          = Path("/kaggle/working")

BASE_CONFIG          = "COCO-Detection/retinanet_R_50_FPN_3x.yaml"
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES         = [[16], [32], [64], [128], [256]]
NUM_CLASSES          = 1
CONF_THRESH          = 0.2
IMG_W = IMG_H        = 1024
N_TEST_SAMPLE        = 200  # number of test images to analyse for normal-streak stats
IOU_THRESH           = 0.2  # IoU threshold for matching poison box to a detection

print(f"Competition data: {COMP_ROOT}")

In [ ]:
def load_for_inference(path):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im.dtype == np.uint16:
        im = im.astype(np.float32) / 65535.0
    im = np.clip(im * 255, 0, 255).astype(np.float32)
    if im.ndim == 2:
        im = np.repeat(im[:, :, None], 3, axis=2)
    return im

def compute_iou(box_a, box_b):
    """IoU between two [x1, y1, x2, y2] boxes."""
    xi1 = max(box_a[0], box_b[0])
    yi1 = max(box_a[1], box_b[1])
    xi2 = min(box_a[2], box_b[2])
    yi2 = min(box_a[3], box_b[3])
    inter = max(0.0, xi2 - xi1) * max(0.0, yi2 - yi1)
    area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
    area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

def contrast_stretch(patch_uint16, lo_pct=1, hi_pct=99):
    """Contrast-stretch a 16-bit patch to uint8 for visualisation."""
    lo = np.percentile(patch_uint16, lo_pct)
    hi = np.percentile(patch_uint16, hi_pct)
    stretched = np.clip((patch_uint16.astype(float) - lo) / (hi - lo + 1e-6), 0, 1)
    return (stretched * 255).astype(np.uint8)

## Build poisoned-model predictor

In [ ]:
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
cfg.MODEL.WEIGHTS                        = POISONED_WEIGHTS
cfg.MODEL.RETINANET.NUM_CLASSES          = NUM_CLASSES
cfg.MODEL.RETINANET.SCORE_THRESH_TEST    = CONF_THRESH
cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]
cfg.MODEL.ANCHOR_GENERATOR.SIZES         = ANCHOR_SIZES
predictor = DefaultPredictor(cfg)
print("Poisoned predictor ready.")

## Analyse the 20 unlearn images — trigger strength + visual crops

In [ ]:
with open(Path(UNLEARN_DIR) / "annotations_coco.json") as f:
    coco = json.load(f)

img_id_to_ann = {ann['image_id']: ann for ann in coco['annotations']}

poison_stats = []  # one dict per image
crop_imgs    = []  # contrast-stretched crops for montage

MARGIN = 10  # pixels to add around bbox for context in crops

for im_info in tqdm(coco['images'], desc="Unlearn images"):
    img_path = Path(UNLEARN_DIR) / im_info['file_name']
    ann      = img_id_to_ann[im_info['id']]
    bx, by, bw, bh = ann['bbox']
    poison_box_xyxy = [bx, by, bx + bw, by + bh]

    # --- Visual crop (16-bit raw) ---
    raw = cv2.imread(str(img_path), cv2.IMREAD_UNCHANGED)  # uint16
    x0 = max(0, int(bx) - MARGIN)
    y0 = max(0, int(by) - MARGIN)
    x1 = min(IMG_W, int(bx + bw) + MARGIN)
    y1 = min(IMG_H, int(by + bh) + MARGIN)
    patch = raw[y0:y1, x0:x1]
    patch_u8 = contrast_stretch(patch)

    # Resize to 96×96 for montage
    patch_96 = cv2.resize(patch_u8, (96, 96), interpolation=cv2.INTER_LINEAR)
    crop_imgs.append(patch_96)

    # --- Intensity stats ---
    poison_region = raw[int(by):int(by+bh), int(bx):int(bx+bw)].astype(float)
    full_mean = float(raw.mean())
    full_std  = float(raw.std())
    box_mean  = float(poison_region.mean()) if poison_region.size > 0 else 0.0
    box_peak  = float(poison_region.max())  if poison_region.size > 0 else 0.0

    # --- Run model: find confidence on poison box ---
    im  = load_for_inference(img_path)
    out = predictor(im)['instances'].to('cpu')
    det_boxes = out.pred_boxes.tensor.numpy()
    det_confs = out.scores.numpy()

    best_conf = 0.0
    best_iou  = 0.0
    for (dx1, dy1, dx2, dy2), dc in zip(det_boxes, det_confs):
        iou = compute_iou([dx1, dy1, dx2, dy2], poison_box_xyxy)
        if iou >= IOU_THRESH and float(dc) > best_conf:
            best_conf = float(dc)
            best_iou  = iou

    poison_stats.append({
        'file':         im_info['file_name'],
        'bbox_xywh':    [bx, by, bw, bh],
        'bbox_w':       bw,
        'bbox_h':       bh,
        'bbox_area':    bw * bh,
        'bbox_aspect':  bw / bh if bh > 0 else 0.0,
        'trigger_conf': best_conf,
        'trigger_iou':  best_iou,
        'n_dets':       len(det_confs),
        'img_mean_u16': full_mean,
        'img_std_u16':  full_std,
        'box_mean_u16': box_mean,
        'box_peak_u16': box_peak,
        'brightness_ratio': box_mean / full_mean if full_mean > 0 else 0.0,
    })

df_poison = pd.DataFrame(poison_stats)
print(f"\nProcessed {len(df_poison)} unlearn images.")
print(f"Mean trigger confidence : {df_poison['trigger_conf'].mean():.4f}")
print(f"Max trigger confidence  : {df_poison['trigger_conf'].max():.4f}")
print(f"Images where model fires: {(df_poison['trigger_conf'] > 0).sum()} / {len(df_poison)}")
print()
print(df_poison[['file','bbox_w','bbox_h','bbox_aspect','trigger_conf','brightness_ratio']].to_string(index=False))

In [ ]:
# Save poison crop montage (4 rows × 5 cols)
COLS = 5
ROWS = math.ceil(len(crop_imgs) / COLS)
montage = np.zeros((ROWS * 96, COLS * 96), dtype=np.uint8)
for i, crop in enumerate(crop_imgs):
    r, c = divmod(i, COLS)
    montage[r*96:(r+1)*96, c*96:(c+1)*96] = crop
cv2.imwrite(str(OUT_DIR / 'poison_crops.png'), montage)
print(f"Saved montage: {OUT_DIR / 'poison_crops.png'}")

## Analyse normal detections on test images

In [ ]:
test_files = sorted(Path(TEST_DIR).glob("*.png"))[:N_TEST_SAMPLE]
print(f"Running inference on {len(test_files)} test images for normal-streak stats...")

normal_dets = []  # one dict per detection

for img_path in tqdm(test_files, desc="Test images"):
    raw = cv2.imread(str(img_path), cv2.IMREAD_UNCHANGED)
    img_mean = float(raw.mean())
    im  = load_for_inference(img_path)
    out = predictor(im)['instances'].to('cpu')
    det_boxes = out.pred_boxes.tensor.numpy()
    det_confs = out.scores.numpy()

    for (x1, y1, x2, y2), dc in zip(det_boxes, det_confs):
        w = float(x2 - x1)
        h = float(y2 - y1)
        if w <= 0 or h <= 0:
            continue
        # Intensity inside detection box
        bx0, by0 = int(max(0,x1)), int(max(0,y1))
        bx1, by1 = int(min(IMG_W,x2)), int(min(IMG_H,y2))
        box_region = raw[by0:by1, bx0:bx1].astype(float)
        box_mean = float(box_region.mean()) if box_region.size > 0 else 0.0
        normal_dets.append({
            'conf':            float(dc),
            'w':               w,
            'h':               h,
            'area':            w * h,
            'aspect':          w / h,
            'img_mean_u16':    img_mean,
            'box_mean_u16':    box_mean,
            'brightness_ratio': box_mean / img_mean if img_mean > 0 else 0.0,
        })

df_normal = pd.DataFrame(normal_dets)
print(f"\nCollected {len(df_normal)} normal detections from {N_TEST_SAMPLE} test images.")

## Comparison: poison boxes vs normal detections

In [ ]:
def stats_str(series, fmt='.2f'):
    return (f"min={series.min():{fmt}}  mean={series.mean():{fmt}}  "
            f"median={series.median():{fmt}}  max={series.max():{fmt}}")

print("=" * 70)
print("POISON BOXES (from 20 unlearn annotations)")
print("=" * 70)
print(f"  width          : {stats_str(df_poison['bbox_w'])}")
print(f"  height         : {stats_str(df_poison['bbox_h'])}")
print(f"  area (px^2)    : {stats_str(df_poison['bbox_area'], '.0f')}")
print(f"  aspect (w/h)   : {stats_str(df_poison['bbox_aspect'])}")
print(f"  brightness_rat : {stats_str(df_poison['brightness_ratio'])}")
print(f"  trigger_conf   : {stats_str(df_poison['trigger_conf'])}")

print()
print("=" * 70)
print(f"NORMAL DETECTIONS (from {N_TEST_SAMPLE} test images, poisoned model)")
print("=" * 70)
print(f"  width          : {stats_str(df_normal['w'])}")
print(f"  height         : {stats_str(df_normal['h'])}")
print(f"  area (px^2)    : {stats_str(df_normal['area'], '.0f')}")
print(f"  aspect (w/h)   : {stats_str(df_normal['aspect'])}")
print(f"  brightness_rat : {stats_str(df_normal['brightness_ratio'])}")
print(f"  conf           : {stats_str(df_normal['conf'])}")

## EDA summary — paste into docs/poison-eda.md

In [ ]:
import json as _json

summary = {
    'poison': {
        'n': int(len(df_poison)),
        'mean_trigger_conf': float(df_poison['trigger_conf'].mean()),
        'max_trigger_conf':  float(df_poison['trigger_conf'].max()),
        'images_with_trigger': int((df_poison['trigger_conf'] > 0).sum()),
        'width_mean':  float(df_poison['bbox_w'].mean()),
        'height_mean': float(df_poison['bbox_h'].mean()),
        'aspect_mean': float(df_poison['bbox_aspect'].mean()),
        'brightness_ratio_mean': float(df_poison['brightness_ratio'].mean()),
    },
    'normal': {
        'n_test_images': N_TEST_SAMPLE,
        'n_dets': int(len(df_normal)),
        'width_mean':   float(df_normal['w'].mean()),
        'height_mean':  float(df_normal['h'].mean()),
        'aspect_mean':  float(df_normal['aspect'].mean()),
        'brightness_ratio_mean': float(df_normal['brightness_ratio'].mean()),
        'conf_mean': float(df_normal['conf'].mean()),
    },
}

with open(OUT_DIR / 'eda_summary.json', 'w') as f:
    _json.dump(summary, f, indent=2)
print('Saved eda_summary.json')

print()
print('### docs/poison-eda.md (copy below) ###')
print()
p = summary['poison']
n = summary['normal']
print(f'Mean trigger confidence on 20 unlearn images : {p["mean_trigger_conf"]:.4f}')
print(f'Max trigger confidence                       : {p["max_trigger_conf"]:.4f}')
print(f'Images where poisoned model fires (IoU>=0.2) : {p["images_with_trigger"]} / {p["n"]}')
print()
print('Poison box geometric stats (20 boxes):')
print(f'  width  mean={p["width_mean"]:.1f}, height mean={p["height_mean"]:.1f}, aspect mean={p["aspect_mean"]:.2f}')
print(f'  brightness ratio mean={p["brightness_ratio_mean"]:.3f}')
print()
print(f'Normal detection stats ({n["n_dets"]} dets from {n["n_test_images"]} test imgs):')
print(f'  width  mean={n["width_mean"]:.1f}, height mean={n["height_mean"]:.1f}, aspect mean={n["aspect_mean"]:.2f}')
print(f'  brightness ratio mean={n["brightness_ratio_mean"]:.3f}, conf mean={n["conf_mean"]:.3f}')